In [1]:
import numpy as np
import pandas as pd
import ml_utils as mu

ligands = pd.read_csv("../data/dekois_ache/ligands.csv")
scores = pd.read_csv("../results/docking_scores.csv")

df = ligands.merge(scores[["id", "score"]], on="id").reset_index(drop=True)
print("compounds:", df.shape)
print(df["label"].value_counts().rename({0: "decoys", 1: "actives"}))
df.head(3)

compounds: (1219, 11)
label
decoys     1179
actives      40
Name: count, dtype: int64


,id,smiles,label,MW,logP,HBD,HBA,RotB,charge,heavy,score
0,BDB9012,c1cccc(c12)[nH]cc2CCNC(=O)CCCCCCNc3c(CCCC4)c4[...,1,538.543,7.0421,3,2,11,1,37,-12.516
1,BDB50318731,c1cccc(c12)[nH]cc2CCC(=O)NCCCCCCNc3c(CCCC4)c4[...,1,469.653,5.7353,3,2,11,1,35,-12.060
2,BDB8965,c1cccc(c12)[nH+]c3c(CCCC3)c2NCCC[NH+](C)CCCNc4...,1,510.750,4.1977,3,2,10,3,38,-11.856


In [2]:
X_morgan = mu.morgan_matrix(df["smiles"])
print("Morgan matrix:", X_morgan.shape)
print("average bits set per molecule:", round(X_morgan.sum(axis=1).mean(), 1))

Morgan matrix: (1219, 2048)
average bits set per molecule: 47.5


In [3]:
df["scaffold"] = mu.murcko_scaffolds(df["smiles"])
print("unique scaffolds:", df["scaffold"].nunique())
print("scaffolds carrying an active:", df.loc[df.label == 1, "scaffold"].nunique())
print("\nlargest scaffold groups:")
print(df["scaffold"].value_counts().head(5))

unique scaffolds: 1065
scaffolds carrying an active: 32

largest scaffold groups:
scaffold
c1ccccc1                                                      14
O=C(CSc1nnc(CNc2ccccc2)n1-c1ccccc1)Nc1ccccc1                   5
c1ccc(C(CCN2CC[NH+](CCC(c3ccccc3)c3ccccc3)CC2)c2ccccc2)cc1     5
c1ccc(Cc2ccccc2)cc1                                            5
O=C(CSc1nnc(CNc2ccccc2)[nH]1)Nc1ccccc1                         4
Name: count, dtype: int64


In [5]:
from rdkit import Chem

POSE_DIR = "../results/poses"

for cid in df["id"].head(5):
    m = mu.top_pose(f"{POSE_DIR}/{cid}_out.pdbqt", name=cid)
    print(f"{cid:16s} {m.GetNumAtoms():3d} atoms   {m.GetNumConformers()} conformer   {Chem.MolToSmiles(Chem.RemoveHs(m))[:45]}")

BDB9012           72 atoms   1 conformer   O=C(CCCCCCNc1c2c([nH+]c3cc(Cl)cc(Cl)c13)CCCC2
BDB50318731       72 atoms   1 conformer   O=C(CCc1c[nH]c2ccccc12)NCCCCCCNc1c2c([nH+]c3c
BDB8965           82 atoms   1 conformer   C[NH+](CCCNc1c2c([nH+]c3ccccc13)CCCC2)CCCNc1c
BDB9054           79 atoms   1 conformer   Clc1ccc2c(NCCCCCCCNc3c4c([nH+]c5cc(Cl)ccc35)C
BDB9035           72 atoms   1 conformer   O=C(Cc1c[nH]c2ccccc12)NCCCCCCCNc1c2c([nH+]c3c


In [6]:
SDF_OUT = "../prep/top_poses.sdf"

writer = Chem.SDWriter(SDF_OUT)
kept, failed = [], []

for n, cid in enumerate(df["id"], start=1):
    try:
        writer.write(mu.top_pose(f"{POSE_DIR}/{cid}_out.pdbqt", name=cid))
        kept.append(cid)
    except Exception as error:
        failed.append((cid, str(error)))
    if n % 200 == 0:
        print(n, "poses processed")

writer.close()
print("\nwritten:", len(kept), "  failed:", len(failed))
if failed:
    print(failed[:5])

200 poses processed
400 poses processed
600 poses processed
800 poses processed
1000 poses processed
1200 poses processed

written: 1219   failed: 0


In [7]:
receptor = mu.load_receptor("../prep/receptor_A_H.pdb")
print("receptor residues:", len(receptor.residues))

/home/shams/miniconda3/envs/ache-docking/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/shams/miniconda3/envs/ache-docking/lib/python3.11/site-packages/MDAnalysis/topology/tables.py:52: DeprecationWarning: Deprecated in version 2.8.0
MDAnalysis.topology.tables has been moved to MDAnalysis.guesser.tables. This import point will be removed in MDAnalysis version 3.0.0
  warnings.warn(wmsg, category=DeprecationWarning)


receptor residues: 530


In [8]:
ifp = mu.ifp_matrix("../prep/top_poses.sdf", receptor)
ifp.index = kept

print("IFP matrix:", ifp.shape)
print("\nmost frequently contacted residues and interaction types:")
print(ifp.sum().sort_values(ascending=False).head(12))

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 1219/1219 [00:31<00:00, 38.64it/s]


IFP matrix: (1219, 115)

most frequently contacted residues and interaction types:
TYR124.A|VdWContact     1044
TYR341.A|Hydrophobic    1015
PHE338.A|Hydrophobic     941
PHE338.A|VdWContact      910
TRP286.A|Hydrophobic     880
TYR341.A|VdWContact      877
TRP86.A|Hydrophobic      810
TRP86.A|VdWContact       800
TYR337.A|Hydrophobic     700
HIS447.A|VdWContact      663
TYR337.A|VdWContact      655
PHE295.A|VdWContact      637
dtype: uint64


In [9]:
import os

OUT = "../results/ml"
os.makedirs(OUT, exist_ok=True)

assert list(df["id"]) == list(kept), "row order of features and labels does not match"

np.savez_compressed(f"{OUT}/morgan.npz", X=X_morgan, ids=df["id"].to_numpy())
ifp.to_csv(f"{OUT}/ifp.csv")
df[["id", "label", "score", "scaffold", "heavy", "RotB", "MW", "logP"]].to_csv(
    f"{OUT}/compounds.csv", index=False)

print("saved to", OUT)
print("  morgan.npz    ", X_morgan.shape)
print("  ifp.csv       ", ifp.shape)
print("  compounds.csv ", df.shape[0], "rows")

saved to ../results/ml
  morgan.npz     (1219, 2048)
  ifp.csv        (1219, 115)
  compounds.csv  1219 rows
